In [1]:
# imports
import pandas as pd
from pathlib import Path

In [5]:
DATA = Path("../data")

beaches = pd.read_csv(DATA / "beach_monthly.csv")
incidents = pd.read_csv(DATA / "incidents.csv")

incidents["month"] = pd.to_datetime(
    incidents["dte_occurred"], dayfirst=True
).dt.strftime("%m/%Y")

keys = ["beach_nm", "month"]

# Count unique incidents
total = (
    incidents.groupby(keys)["incident_id"]
    .nunique()
    .rename("n_incidents")
)

# Count person-level outcomes by severity
severity = (
    incidents.groupby(keys + ["inc_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Mild", "CloseCall", "Severe"], fill_value=0)
    .rename(columns={
        "Mild": "n_mild",
        "CloseCall": "n_closecall",
        "Severe": "n_severe"
    })
)

# Merge targets and predictors
targets = pd.concat([total, severity], axis=1).reset_index()

df = beaches.merge(targets, on=keys, how="left", validate="one_to_one")

target_cols = ["n_incidents", "n_mild", "n_closecall", "n_severe"]
df[target_cols] = df[target_cols].fillna(0).astype(int)

df = df[target_cols + [c for c in df if c not in target_cols]]
# Sort chronologically by month, then beach
df["_date"] = pd.to_datetime(df["month"], format="%m/%Y")
df = df.sort_values(["_date", "beach_nm"]).drop(columns="_date")
df = df.reset_index(drop=True)

df.to_csv(DATA / "task1.csv", index=False)

print(df.shape)
display(df.head())

(9600, 33)


,n_incidents,n_mild,n_closecall,n_severe,beach_nm,postcode,month,visitors,shore_len_m,n_rips,...,flags,patrolled,equip_rating,rocky,fishing,n_bins,rockpool,icecream_visits,surfer_prop,marine_health
0,0,0,0,0,Angourie Beach,2464,09/2016,2382,647,0.0,...,0,0,52,1,1,12,0,1.3,0.489,M
1,0,0,0,0,Aslings Beach (Eden),2551,09/2016,2239,583,2.8,...,1,1,59,0,1,11,1,0.1,0.174,L
2,0,0,0,0,Austinmer Beach,2515,09/2016,14937,568,1.0,...,0,1,72,0,0,20,1,1.4,0.205,L
3,0,0,0,0,Avalon Beach,2107,09/2016,3337,1370,2.3,...,1,1,69,0,0,15,0,2.4,0.575,H
4,0,0,0,0,Avoca Beach,2251,09/2016,9496,846,1.1,...,0,1,47,1,1,19,0,1.7,0.153,H
